# Student Academic Outcome Prediction — Exploratory Data Analysis

This notebook explores the public UCI Student Dropout and Academic Success dataset (ID 697) before model training. The dataset is **not UEAB student data**.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo
from pathlib import Path

OUT = Path('outputs')
OUT.mkdir(exist_ok=True)

## 1. Load the dataset and inspect its structure

In [ ]:
dataset = fetch_ucirepo(id=697)
X = dataset.data.features.copy()
y = dataset.data.targets.iloc[:, 0].astype(str).str.strip()
print('Feature matrix shape:', X.shape)
print('Target length:', len(y))
display(X.head())
display(X.info())

## 2. Target class distribution

Inspect the counts and percentages for Dropout, Enrolled, and Graduate.

In [ ]:
class_counts = y.value_counts().rename_axis('outcome').reset_index(name='records')
class_counts['percentage'] = 100 * class_counts['records'] / len(y)
display(class_counts)
ax = class_counts.sort_values('records').plot.barh(x='outcome', y='records', legend=False, figsize=(8, 4), title='Student outcome class distribution')
ax.set_xlabel('Records')
plt.tight_layout()
plt.show()

## 3. Data quality and feature profile

In [ ]:
profile = pd.DataFrame({
    'data_type': X.dtypes.astype(str),
    'unique_values': X.nunique(dropna=True),
    'missing_values': X.isna().sum(),
    'missing_percentage': 100 * X.isna().mean()
})
display(profile)
display(X.describe().T)

## 4. Correlation analysis for quantitative features

Only measured quantities, grades, curriculum counts and macroeconomic variables are included. Integer-coded nominal categories such as course, nationality, and occupation are excluded because their codes do not represent meaningful numeric distances. Correlation does not imply causation.

In [ ]:
quantitative_features = [
    'Previous qualification (grade)', 'Admission grade', 'Age at enrollment',
    'Curricular units 1st sem (credited)', 'Curricular units 1st sem (enrolled)',
    'Curricular units 1st sem (evaluations)', 'Curricular units 1st sem (approved)',
    'Curricular units 1st sem (grade)', 'Curricular units 1st sem (without evaluations)',
    'Curricular units 2nd sem (credited)', 'Curricular units 2nd sem (enrolled)',
    'Curricular units 2nd sem (evaluations)', 'Curricular units 2nd sem (approved)',
    'Curricular units 2nd sem (grade)', 'Curricular units 2nd sem (without evaluations)',
    'Unemployment rate', 'Inflation rate', 'GDP'
]
quantitative_features = [c for c in quantitative_features if c in X.columns]
corr = X[quantitative_features].corr(method='pearson')
display(corr.round(2))
fig, ax = plt.subplots(figsize=(13, 10))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=75, ha='right', fontsize=7)
ax.set_yticklabels(corr.columns, fontsize=7)
ax.set_title('Pearson correlation — quantitative student features')
fig.colorbar(im, ax=ax, label='Correlation')
plt.tight_layout()
plt.show()

## 5. Review model outputs after training

Run `python train_model.py` first. The script generates model comparison, cross-validation, confusion-matrix, and permutation feature-importance outputs.

In [ ]:
comparison_file = OUT / 'model_comparison.csv'
if comparison_file.exists():
    display(pd.read_csv(comparison_file))
else:
    print('Run python train_model.py to generate model evaluation outputs.')

importance_file = OUT / 'feature_importance.csv'
if importance_file.exists():
    display(pd.read_csv(importance_file).head(15))